# 00 · Configuración, fuentes y descarga oficial

**Pregunta.** ¿Las seis bases federales 2009–2024 y el manual oficial pueden obtenerse, validarse e inventariarse de forma reproducible?

**Entradas.** `configs/sources.yaml`, página oficial del INE y documentos locales de referencia. Las unidades de esta fase son archivos y miembros de archivo, no personas ni territorios.

**Método.** Se usa HTTP con *streaming*, archivo temporal, SHA-256, validación de firma/contenido, CRC y rutas seguras. Los ZIP crudos quedan inmutables en `data/raw/`; su extracción separada queda en `data/interim/extracted/`.

**Criterios.** Deben existir seis ZIP federales válidos, un PDF oficial, un manifiesto con URL solicitada/final y un inventario de miembros. Voto anticipado, voto en prisión preventiva y Poder Judicial se registran como universos separados.

**Salidas.** `data/manifests/source_manifest.csv`, `data/manifests/zip_member_inventory.csv` y carpetas extraídas. Tiempo esperado: varios minutos; descarga aproximada: 292 MB comprimidos más el manual.

## Entorno y raíz del proyecto

Se localiza la raíz mediante archivos versionados y se muestran versiones efectivas. No se usa una ruta personal absoluta ni estado de otro notebook.

In [ ]:
from pathlib import Path
import hashlib
import platform
import sys
import time

import pandas as pd
import yaml
from IPython.display import Markdown, display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "configs/sources.yaml").exists():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del proyecto")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RUN_STARTED = time.perf_counter()
environment = {
    "project_root_detected": PROJECT_ROOT.name,
    "python": sys.version.split()[0],
    "implementation": platform.python_implementation(),
    "platform": platform.platform(),
    "pandas": pd.__version__,
}
pd.Series(environment, name="valor").to_frame()

In [ ]:
display(Markdown(
    f"**Interpretación.** La ejecución usa Python `{environment['python']}` y detectó la raíz "
    f"`{environment['project_root_detected']}` sin depender de una ruta absoluta escrita en el notebook."
))

## Registro de alcance y fuentes

Antes de transferir bytes se inspecciona la configuración. El universo principal incluye sólo elecciones federales comparables del portal; los procesos especiales se documentan y se excluyen de esta descarga principal.

In [ ]:
with (PROJECT_ROOT / "configs/sources.yaml").open(encoding="utf-8") as stream:
    sources = yaml.safe_load(stream)

source_rows = []
for year, item in sorted(sources["elections"].items(), key=lambda pair: int(pair[0])):
    source_rows.append({"source_id": f"ccpc_federal_{year}", "year": int(year), **item})
configured_sources = pd.DataFrame(source_rows)
display(configured_sources[["source_id", "year", "kind", "filename", "url"]])
display(pd.Series(sources["scope"], name="definición"))

In [ ]:
excluded = sources["scope"]["excluded_from_main_panel"]
display(Markdown(
    f"**Interpretación.** Se configuraron `{len(configured_sources)}` elecciones federales. "
    f"Permanecen fuera del panel principal `{len(excluded)}` universos: " + "; ".join(excluded) + "."
))

## Trazabilidad de documentos locales

Se calculan hashes de la convocatoria, el manual y los dos documentos metodológicos existentes. Esto conserva la procedencia de las versiones realmente leídas; no modifica sus originales.

In [ ]:
def sha256_local(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


document_paths = [
    PROJECT_ROOT / "Convocatoria_Hackaton-1.pdf",
    PROJECT_ROOT / "Manual-Conteos-Censales-2009-2024.pdf",
    PROJECT_ROOT / "RADAR27_Plan_Maestro_Agente.md",
    PROJECT_ROOT / "RADAR27_Propuesta_Equipo_Obsidian.md",
]
document_registry = pd.DataFrame(
    {
        "path": [path.relative_to(PROJECT_ROOT).as_posix() for path in document_paths],
        "bytes": [path.stat().st_size for path in document_paths],
        "sha256": [sha256_local(path) for path in document_paths],
    }
)
display(document_registry)

In [ ]:
display(Markdown(
    f"**Interpretación.** Se registraron `{len(document_registry)}` documentos locales. "
    "Sus hashes identifican exactamente las versiones consultadas en esta fase."
))

## Descarga, validación y extracción segura

La llamada siguiente muestra explícitamente el archivo de configuración, el destino y los controles aplicados por la utilidad versionada `src/radar27/ingest/download_ccpc.py`: HTTP con redirecciones registradas, escritura `.part`, SHA-256, rechazo de HTML, prueba CRC, prohibición de `..`, rutas absolutas y enlaces simbólicos, y extracción separada. Si una instantánea existente tiene contenido distinto, la ejecución se detiene.

In [ ]:
from radar27.ingest.download_ccpc import download_configured_sources

download_started = time.perf_counter()
manifest_rows, member_rows = download_configured_sources(
    PROJECT_ROOT,
    notebook="notebooks/00_setup_sources_download.ipynb",
    cell_id="download-official-sources",
    extract=True,
)
download_seconds = time.perf_counter() - download_started
manifest = pd.DataFrame(manifest_rows)
members = pd.DataFrame(member_rows)
display(manifest[[
    "source_id", "year", "http_status", "content_type", "bytes", "sha256",
    "requested_url", "final_url", "validation_status"
]])

In [ ]:
zip_rows = manifest[manifest["kind"].eq("zip")]
valid_zip_count = int(zip_rows["validation_status"].eq("zip_signature_crc_paths_verified").sum())
requested_equals_final = int((manifest["requested_url"] == manifest["final_url"]).sum())
display(Markdown(
    f"**Resultado.** Se validaron `{valid_zip_count}` de `{len(zip_rows)}` ZIP mediante firma, CRC y rutas "
    f"seguras. La descarga tardó `{download_seconds:.1f}` s. En `{requested_equals_final}` de "
    f"`{len(manifest)}` fuentes la URL final coincidió exactamente con la solicitada; cualquier "
    "redirección restante quedó en el manifiesto."
))

## Inventario interno y tamaño expandido

Se resume cada archivo comprimido antes de interpretar tablas. El conteo incluye miembros documentales y de datos; todavía no se impone una unidad de observación.

In [ ]:
member_summary = (
    members.groupby("year", as_index=False)
    .agg(
        members=("member_name", "count"),
        files=("is_dir", lambda values: int((~values.astype(bool)).sum())),
        compressed_bytes=("compressed_bytes", "sum"),
        uncompressed_bytes=("uncompressed_bytes", "sum"),
    )
)
member_summary["expansion_ratio"] = (
    member_summary["uncompressed_bytes"] / member_summary["compressed_bytes"].replace(0, pd.NA)
)
display(member_summary)
display(members[["year", "member_name", "uncompressed_bytes", "crc32"]])

In [ ]:
display(Markdown(
    f"**Resultado.** Los seis ZIP contienen `{len(members)}` miembros registrados y "
    f"`{int(member_summary['uncompressed_bytes'].sum()):,}` bytes declarados al expandirse. "
    "La granularidad y compatibilidad de cada tabla se auditan en el notebook 01; este inventario "
    "no supone que nombres parecidos tengan el mismo significado."
))

## Cierre de la fase de descarga

Se comprueban los artefactos mínimos y se informa duración y límites. Tener ZIP íntegros no implica que sus esquemas sean comparables ni que la cartografía histórica esté armonizada.

In [ ]:
required_outputs = [
    PROJECT_ROOT / "data/manifests/source_manifest.csv",
    PROJECT_ROOT / "data/manifests/zip_member_inventory.csv",
]
checks = {
    "six_election_zips": len(zip_rows) == 6,
    "all_http_200": bool(manifest["http_status"].eq(200).all()),
    "all_outputs_exist": all(path.exists() for path in required_outputs),
    "all_years_extracted": all(
        (PROJECT_ROOT / "data/interim/extracted" / str(year)).exists()
        for year in configured_sources["year"]
    ),
}
elapsed = time.perf_counter() - RUN_STARTED
display(pd.Series(checks, name="aprobado").to_frame())
display(Markdown(
    f"**Cierre.** `{sum(checks.values())}/{len(checks)}` comprobaciones pasaron en `{elapsed:.1f}` s. "
    "Siguiente paso: determinar formatos, esquemas, categorías y granularidad reales sin unir años. "
    "La cartografía oficial de mayo de 2023 está identificada, pero su compatibilidad con cada ZIP "
    "permanece pendiente y por ello no se supera todavía el Gate 1 completo del plan maestro."
))